## 0)  Project Root

In [1]:
import sys
from pathlib import Path

def get_project_root(project_dir_name="Project", marker=".git"):
    """
    1) Walk upwards until we find the repo root (contains .git).
    2) Return <repo_root>/<project_dir_name> as the actual project root
       (the folder that contains models/, training/, notebooks/, etc.).
    3) Add that project root to sys.path for imports.
    """
    current = Path.cwd().resolve()

    # Step 1: find repo root by .git
    repo_root = None
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            repo_root = parent
            break

    if repo_root is None:
        raise RuntimeError(f"Repo root not found (no '{marker}' directory)")

    # Step 2: define actual project root
    project_root = repo_root / project_dir_name
    if not project_root.exists():
        raise RuntimeError(
            f"Found repo root at {repo_root}, but '{project_dir_name}' folder not found."
        )

    # Step 3: add to sys.path
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f"[OK] Repo root:    {repo_root}")
    print(f"[OK] Project root: {project_root}")
    return project_root

PROJECT_ROOT = get_project_root()



[OK] Repo root:    /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning
[OK] Project root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project


In [2]:
from pathlib import Path
import json
import random

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Optional: stronger reproducibility (can slow down / error on some ops)
# torch.use_deterministic_algorithms(True)

# CUDA deterministic settings (only matters on CUDA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


## Load precomputed train/test sequences (*)

In [3]:
# ---------------------------------------------------------
# Load precomputed train/test sequences
# ---------------------------------------------------------
# Earlier in the pipeline we generated:
#   - X_train.npy : training input sequences
#   - y_train.npy : training labels
#   - X_test.npy  : test input sequences
#   - y_test.npy  : test labels
#   - meta.json   : metadata (feature names, sequence length, etc.)
#
# We now simply load these NumPy arrays back into memory so the LSTM
# training script can use them directly.
#
# NOTE:
#   • Using .npy keeps array shapes and dtypes intact.
#   • This avoids recomputing sequences every time we train.
#   • The directory 03_Sequences acts as a reproducible “data build” stage.
# ---------------------------------------------------------

######################################################################
# ---------------------------------------------------------
# Select sequence length (must match precomputed data)
# ---------------------------------------------------------
SEQ_LEN = 60   # <-- change this to 30, 120, etc.
print("Using SEQ_LEN =", SEQ_LEN)

######################################################################

# Directory containing the saved sequences and metadata
SEQ_DIR = PROJECT_ROOT / "03_Sequences" / f"seq{SEQ_LEN}"

# Paths to the saved NumPy arrays and metadata file
X_train_path = SEQ_DIR / "X_train.npy"
y_train_path = SEQ_DIR / "y_train.npy"
X_test_path  = SEQ_DIR / "X_test.npy"
y_test_path  = SEQ_DIR / "y_test.npy"
meta_path    = SEQ_DIR / "meta.json"

print("Using:")
print("  ", X_train_path)
print("  ", y_train_path)
print("  ", X_test_path)
print("  ", y_test_path)
print("  ", meta_path)

# ---------------------------------------------------------
# Load the datasets into memory
# ---------------------------------------------------------
# These arrays have the shapes:
#   X_train : (num_train_samples, seq_len, num_features)
#   y_train : (num_train_samples,)
#   X_test  : (num_test_samples,  seq_len, num_features)
#   y_test  : (num_test_samples,)
#
# They contain exactly the same data produced previously in the
# feature-building + sequence-building notebook.
# ---------------------------------------------------------
X_train = np.load(X_train_path)
y_train = np.load(y_train_path)
X_test  = np.load(X_test_path)
y_test  = np.load(y_test_path)

# ---------------------------------------------------------
# Load metadata including:
#   - feature_cols (list of input features)
#   - seq_len      (window length, e.g. 60)
#   - scaler params (if you saved them)
#   - timestamps or any other contextual info
#
# This ensures the training script “knows” how the sequences were built.
# ---------------------------------------------------------
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

# Show shapes to verify everything is loaded correctly
print("meta:", meta)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


Using SEQ_LEN = 60
Using:
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/seq60/X_train.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/seq60/y_train.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/seq60/X_test.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/seq60/y_test.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/seq60/meta.json
meta: {'seq_len': 60, 'feature_cols': ['log_ret_1m', 'log_ret_5m', 'log_ret_15m', 'vol_5m', 'vol_15m', 'range_pct_1m', 'range_pct_5m', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'body_norm

## Create a validation set from the *end* of the training set

In [4]:
# -------------------------------------------------------------
# Create a validation set from the *end* of the training set
# -------------------------------------------------------------
# We already performed a strict time-based split:
#   • Training data = earlier part of the dataset
#   • Test data     = later, unseen part
#
# Now we take ~10% of the *training* portion and reserve it as
# a validation set to monitor model performance DURING training.
#
# IMPORTANT — Time Series Rule:
#   We DO NOT shuffle data or pick random samples.
#   Validation must also contain LATER time periods than training.
#
# So we do:
#   - The FIRST (90%) of X_train → X_train_final
#   - The LAST  (10%) of X_train → X_val
#
# This preserves correct temporal ordering and prevents future leakage.
# -------------------------------------------------------------
val_ratio = 0.1
n_train = X_train.shape[0]              # total number of pre-test training samples
val_size = max(1, int(n_train * val_ratio)) # ensure at least 1 sample

# -------------------------------------------------------------
# Final training set:
#   All training samples except the last 'val_size'
# -------------------------------------------------------------
X_train_final = X_train[:-val_size]
y_train_final = y_train[:-val_size]

# -------------------------------------------------------------
# Validation set:
#   The LAST 'val_size' samples from the training data
#   (chronologically later than the training set)
# -------------------------------------------------------------
X_val = X_train[-val_size:]
y_val = y_train[-val_size:]

print("Train final:", X_train_final.shape, y_train_final.shape)
print("Val        :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)


Train final: (1704447, 60, 15) (1704447,)
Val        : (189382, 60, 15) (189382,)
Test       : (473458, 60, 15) (473458,)


## Feature Scaling (StandardScaler)

In [ ]:
import hashlib

# -------------------------------------------------------------
# Feature Scaling (StandardScaler)
# -------------------------------------------------------------
# We normalize each feature so that:
#   • mean ≈ 0
#   • standard deviation ≈ 1
#
# This greatly stabilizes LSTM training and ensures that
# features with large numeric ranges do NOT dominate the model.
#
# IMPORTANT (NO DATA LEAKAGE):
#   The scaler is fit ONLY on the TRAINING DATA.
#   Validation and test data are transformed using the SAME scaler.
#   This prevents exposing the model to future information.
#
# Since the data has shape:
#   (num_samples, seq_len, num_features)
#
# We:
#   1) Flatten the time dimension (seq_len) into samples
#   2) Fit the scaler over ALL time steps in the training set
#   3) Apply the scaler to train/val/test and reshape back
# -------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from utils.data_utils import apply_scaler_3d

num_features = X_train_final.shape[2]   # number of features per time step

# Create StandardScaler (mean=0, std=1)
scaler = StandardScaler()

# -------------------------------------------------------------
# Fit scaler ONLY on the training data
# -------------------------------------------------------------
# reshape (N, SEQ_LEN, num_features) → (N * SEQ_LEN, num_features)
# So we treat every time step as an independent sample for scaling.
X_train_2d = X_train_final.reshape(-1, num_features)

# Compute mean and std for each feature using ONLY training data
scaler.fit(X_train_2d)


# -------------------------------------------------------------
# Apply scaler to all splits
# -------------------------------------------------------------
X_train_scaled = apply_scaler_3d(X_train_final, scaler)
X_val_scaled   = apply_scaler_3d(X_val, scaler)
X_test_scaled  = apply_scaler_3d(X_test, scaler)


# Final sanity check
print("Scaled shapes:")
print("  train:", X_train_scaled.shape)
print("  val  :", X_val_scaled.shape)
print("  test :", X_test_scaled.shape)

def md5_np(a): 
    return hashlib.md5(np.ascontiguousarray(a).tobytes()).hexdigest()

print("RAW X md5:", md5_np(X_test))   # or X_test if you loaded raw
print("SCALED test md5:", md5_np(X_test_scaled))
print("RAW y md5:", md5_np(y_test))
print("RAW shape:", X_test_scaled.shape, y_test.shape)


In [6]:
from utils.data_utils import TimeSeriesDataset

# -------------------------------------------------------------
# Create training, validation, and test Datasets
# -------------------------------------------------------------
batch_size = 16

train_ds = TimeSeriesDataset(X_train_scaled, y_train_final)
val_ds   = TimeSeriesDataset(X_val_scaled,   y_val)
test_ds  = TimeSeriesDataset(X_test_scaled,  y_test)

# -------------------------------------------------------------
# Wrap datasets in DataLoaders (Time-Series Safe)
# -------------------------------------------------------------
# We keep shuffle=False for ALL loaders.
#
# Why:
# - This is temporally ordered financial time-series data.
# - Shuffling would mix different time periods/regimes within an epoch,
#   which can make the experiment less realistic and harder to interpret.
#
# Note:
# - We already do a strict time-based split (train -> val -> test).
# - Using shuffle=False preserves chronological order inside each split.
# -------------------------------------------------------------
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

# Show dataset sizes
# len(train_ds), len(val_ds), len(test_ds)

Xb, yb = next(iter(test_loader))
print("BATCH0 X md5 (old):", md5_np(Xb.detach().cpu().numpy()))
print("BATCH0 y md5 (old):", md5_np(yb.detach().cpu().numpy()))


NameError: name 'X_train_scaled' is not defined

In [14]:
# -------------------------------------------------------------
# Device selection (CPU / CUDA / MPS)
# -------------------------------------------------------------
# We automatically pick the "best" available device:
#   1) Apple Silicon GPU (MPS) if available
#   2) NVIDIA GPU (CUDA) if available
#   3) Fallback to CPU otherwise
#
# This way the same code runs efficiently on different machines.
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: CUDA")
else:
    device = torch.device("cpu")
    print("Using device: CPU")


# -------------------------------------------------------------
# Basic shape information (for sanity + model config)
# -------------------------------------------------------------
# X_train_scaled has shape: (num_train_samples, seq_len, num_features)
# We read seq_len and num_features directly from the data to avoid
# hard-coding them.
# -------------------------------------------------------------
seq_len = X_train_scaled.shape[1]
num_features = X_train_scaled.shape[2]
print(f"Sequence length: {seq_len}, num_features: {num_features}")

Using device: MPS (Apple Silicon GPU)
Sequence length: 60, num_features: 15


##########################################

## TRAINING (*)

In [68]:
from models import LSTMClassifier
from training import train_model
from datetime import datetime, timezone
import json
import joblib
import traceback

EPOCHS = 1
LR = 1e-3

model_kwargs = dict(
    input_size=num_features,
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
)

model = LSTMClassifier(**model_kwargs).to(device)

print("device:", device)
print("len(train_ds):", len(train_ds))
print("len(train_loader):", len(train_loader))
print("batch_size:", batch_size)

MODEL_NAME = model.__class__.__name__
TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S_UTC")

RUN_NAME = f"{MODEL_NAME}_seq{SEQ_LEN}_bs{batch_size}_{TIMESTAMP}"
RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Save scaler early
joblib.dump(scaler, RUN_DIR / "scaler.joblib")

config = {
    "model_class": MODEL_NAME,
    "model_kwargs": model_kwargs,

    # data / sequence configuration
    "seq_len": SEQ_LEN,

    "train": {
        "batch_size": batch_size,
        "epochs_planned": EPOCHS,
        "epochs_completed": 0,
        "learning_rate": {"value": LR, "repr": f"{LR:.0e}"},
        "seed": SEED,
        "device": str(device),
    },

    # bookkeeping
    "timestamp": TIMESTAMP,
    "status": "running",

    # manual annotations (important for experiments)
    "notes": "",
}

######################################################################
config["notes"] = (
    "Set shuffle=True in train_loader for better training."
)
######################################################################

(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

history = None

try:
    history = train_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=EPOCHS,
        lr=LR,
        model_path=str(RUN_DIR / "best_model.pt"),
    )

    # finished successfully
    config["status"] = "finished"

    # infer completed epochs
    if isinstance(history, dict):
        if "train_loss" in history:
            config["train"]["epochs_completed"] = len(history["train_loss"])
        elif "epochs" in history:
            config["train"]["epochs_completed"] = int(history["epochs"])
        else:
            config["train"]["epochs_completed"] = EPOCHS

    # write history first, then DONE
    (RUN_DIR / "history.json").write_text(json.dumps(history, indent=2))
    (RUN_DIR / "DONE").write_text("ok\n")

except Exception as e:
    # mark failure clearly
    config["status"] = "failed"
    (RUN_DIR / "ERROR.txt").write_text(traceback.format_exc())

finally:
    # always rewrite config with final status
    (RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print("Saved run to:", RUN_DIR)

device: mps
len(train_ds): 1704447
len(train_loader): 106528
batch_size: 16


Epochs:   0%|          | 0/1 [00:00<?, ?it/s]


[DEBUG] Starting epoch 1/1
[DEBUG]  Running train_epoch...


Train:   0%|          | 0/106528 [00:00<?, ?it/s]

KeyboardInterrupt: 

##########################################

## TESTING (*)

In [25]:
from models import LSTMClassifier

# -------------------------------------------------------------
# Load the best-performing model checkpoint
# -------------------------------------------------------------
# During training, we saved the model weights (state_dict) every
# time the validation loss improved.  This file contains ONLY the
# trained parameters — not the model class itself.
#
# To load it correctly, we must:
#   1. Recreate the SAME model architecture (same layers/sizes)
#   2. Load the saved state_dict into that model
#   3. Move the model to the correct device (CPU / GPU / MPS)
#
# NOTE:
#   • If ANY model hyperparameter changes (num_features, hidden_size,
#     num_layers, dropout), loading will fail.
#   • This pattern is standard in PyTorch: checkpoint = weights only.
# -------------------------------------------------------------

# -------------------------------------------------------------
# Manually select run folder inside /runs
# -------------------------------------------------------------
RUN_NAME = "LSTMClassifier_seq60_bs16_TrainShuffleTrue_legacy"  # <-- EDIT THIS

RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
assert RUN_DIR.exists(), f"RUN_DIR not found: {RUN_DIR}"

print("Using RUN_DIR:", RUN_DIR)

best_model = LSTMClassifier(
    input_size=num_features, hidden_size=64, num_layers=2, dropout=0.1
)

CKPT_PATH = Path(RUN_DIR) / "best_model.pt"
assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"

# Load weights from file into the model
best_model.load_state_dict(
    torch.load(CKPT_PATH, map_location=device)
)

# Ensure the model runs on the same device as the evaluation tensors
best_model.to(device)

def fp(m):
    h = hashlib.sha256()
    for k in sorted(m.state_dict().keys()):
        t = m.state_dict()[k].detach().cpu().contiguous().numpy()
        h.update(k.encode()); h.update(t.tobytes())
    return h.hexdigest()[:16]

print("MODEL_FP:", fp(best_model))

Using RUN_DIR: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_legacy
MODEL_FP: af2305bdd0a7a91b


In [26]:
from utils.data_utils import make_test_loader
import hashlib

test_loader, X_test_raw, y_test, config = make_test_loader(
    run_dir=RUN_DIR,
    seq_dir=SEQ_DIR,
    shuffle=False
)

## EVAL (Legacy)

In [27]:
# -------------------------------------------------------------
# 2) Accuracy + confusion per t_to_end_min
# -------------------------------------------------------------
# Here we go beyond global metrics and analyze performance
# as a function of "minutes to end of 15-min window".
#
# Steps:
#   1) Find where t_to_end_min lives in the feature vector
#   2) Extract t_to_end_min for each TEST sample
#      (from the *unscaled* X_test, last time step in each sequence)
#   3) Run a forward pass over test_loader to collect:
#         - predicted labels
#         - true labels
#   4) Build a DataFrame and compute:
#         - TP/FP/TN/FN per t_to_end_min
#         - accuracy per t_to_end_min
# -------------------------------------------------------------
import pandas as pd
from tqdm.auto import tqdm

# 2.1) Find the feature index for t_to_end_min
t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
print(f"t_to_end_min is at feature index: {t_to_end_min_idx}")

# 2.2) Extract t_to_end_min from the UN-SCALED test data
#      Each sequence has shape (seq_len, num_features).
#      We take the LAST timestep [-1] for each sample:
#         → this corresponds to the "current" minute the model is predicting for.
t_to_end_min_values = X_test[:, -1, t_to_end_min_idx]  # shape: (num_test,)

print(f"Unique t_to_end_min values: {np.unique(t_to_end_min_values)}")

# 2.3) Collect predictions and true labels from the best model
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(
        tqdm(test_loader, desc="Predicting (test)", leave=False)
    ):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)     # IMPORTANT: use best_model
        probs  = torch.sigmoid(logits)
        preds  = (probs >= 0.5).long()

        if batch_i == 0:
            xb = X_batch.detach().cpu().contiguous().numpy()
            lg = logits.detach().cpu().contiguous().numpy()
            pr = preds.detach().cpu().contiguous().numpy()

            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])

            print("training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)

        # IMPORTANT: extend with numpy arrays, not torch tensors
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_preds  = np.array(all_preds, dtype=int).reshape(-1)
all_labels = np.array(all_labels, dtype=int).reshape(-1)

assert (
    all_preds.shape[0] == t_to_end_min_values.shape[0]
), "Mismatch: number of test predictions != number of t_to_end_min entries"


# 2.4) Build a DataFrame for per-minute analysis
df_results = pd.DataFrame(
    {
        "t_to_end_min": t_to_end_min_values,
        "y_true": all_labels,
        "y_pred": all_preds,
    }
)

# Add confusion components per sample
df_results["tp"] = ((df_results.y_true == 1) & (df_results.y_pred == 1)).astype(int)
df_results["fp"] = ((df_results.y_true == 0) & (df_results.y_pred == 1)).astype(int)
df_results["tn"] = ((df_results.y_true == 0) & (df_results.y_pred == 0)).astype(int)
df_results["fn"] = ((df_results.y_true == 1) & (df_results.y_pred == 0)).astype(int)
df_results["correct"] = (df_results["tp"] + df_results["tn"]).astype(int)

# 2.5) Aggregate stats by t_to_end_min
stats = (
    df_results.groupby("t_to_end_min")
    .agg(
        count=("correct", "count"),
        tp=("tp", "sum"),
        fp=("fp", "sum"),
        tn=("tn", "sum"),
        fn=("fn", "sum"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

stats["accuracy_pct"] = stats["accuracy"] * 100

# 2.6) Print per-minute results
print("\n" + "=" * 70)
print("ACCURACY + CONFUSION METRICS PER t_to_end_min")
print("=" * 70)
print(stats.to_string(index=False))
print("=" * 70)

# 2.7) Baseline vs model
# Baseline (always predicting 1) accuracy = fraction of positives in test labels
baseline_acc = all_labels.mean()
model_acc = (all_preds == all_labels).mean()

print(f"\nBaseline Accuracy (always predict 1): {baseline_acc:.4f}")
print(f"Model Test Accuracy (from preds):      {model_acc:.4f}")

t_to_end_min is at feature index: 14
Unique t_to_end_min values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]


Predicting (test):   0%|          | 0/29592 [00:00<?, ?it/s]

MODEL_FP: af2305bdd0a7a91b
X_FP: bafdebb3d1c993b8
LOGITS_FP: 0be2031792c35dfc
PREDS_FP: 71b60f70dc4df314
training: False
param dtype: torch.float32
X dtype: torch.float32
device: mps:0

ACCURACY + CONFUSION METRICS PER t_to_end_min
 t_to_end_min  count    tp   fp    tn   fn  accuracy  accuracy_pct
          1.0  31564 15433  248 15575  308  0.982385     98.238500
          2.0  31564 14730  958 14865 1011  0.937619     93.761881
          3.0  31564 14179 1511 14312 1562  0.902642     90.264225
          4.0  31564 13724 1921 13902 2017  0.875238     87.523761
          5.0  31564 13305 2401 13422 2436  0.846756     84.675580
          6.0  31564 12915 2807 13016 2826  0.821537     82.153719
          7.0  31564 12581 3183 12640 3160  0.799043     79.904321
          8.0  31564 12107 3628 12195 3634  0.769928     76.992777
          9.0  31564 11633 3994 11829 4108  0.743315     74.331517
         10.0  31564 11108 4462 11361 4633  0.711855     71.185528
         11.0  31564 10680 4918

In [28]:
def hash_array(a: np.ndarray) -> str:
    return hashlib.md5(a.tobytes()).hexdigest()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("X_test hash:", hash_array(X_test))
print("y_test hash:", hash_array(y_test))

X_test shape: (473458, 60, 15)
y_test shape: (473458,)
X_test hash: 821e37fd2e30d0dd30749865b5d2ab94
y_test hash: 7138a0d4459944e41259e786b11688ec


##########################################

## EVAL (NEW)

In [29]:
import numpy as np
from tqdm import tqdm

all_probs  = []
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_i, (X_batch, y_batch) in enumerate(tqdm(test_loader, desc="Predicting (test)", leave=False)):
        X_batch = X_batch.to(device)
        y_batch = y_batch.view(-1).to(device)

        logits = best_model(X_batch).view(-1)                 # ensure shape (batch,)
        probs  = torch.sigmoid(logits)                   # UP probability in [0,1]
        preds  = (probs >= 0.5).long()

        ########################################
        # DEBUG: first batch only
        if batch_i == 0:
            # 1) Model fingerprint (again, right now)
            import hashlib, torch
            def fp(m):
                h = hashlib.sha256()
                for k in sorted(m.state_dict().keys()):
                    t = m.state_dict()[k].detach().cpu().contiguous().numpy()
                    h.update(k.encode()); h.update(t.tobytes())
                return h.hexdigest()[:16]

            print("MODEL_FP:", fp(best_model))

            # 2) Input fingerprint (whole tensor, not just 5 values)
            xb = X_batch.detach().cpu().contiguous().numpy()
            print("X_FP:", hashlib.sha256(xb.tobytes()).hexdigest()[:16])

            # 3) Logits fingerprint
            lg = logits.detach().cpu().contiguous().numpy()
            print("LOGITS_FP:", hashlib.sha256(lg.tobytes()).hexdigest()[:16])

            # 4) Preds fingerprint
            pr = preds.detach().cpu().contiguous().numpy()
            print("PREDS_FP:", hashlib.sha256(pr.tobytes()).hexdigest()[:16])
            
            print("model.training:", best_model.training)
            print("param dtype:", next(best_model.parameters()).dtype)
            print("X dtype:", X_batch.dtype)
            print("device:", next(best_model.parameters()).device)
        ########################################

        all_probs.extend(probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

all_probs  = np.array(all_probs).reshape(-1)
all_preds  = np.array(all_preds).reshape(-1).astype(int)
all_labels = np.array(all_labels).reshape(-1).astype(int)


Predicting (test):   0%|          | 24/29592 [00:00<02:07, 231.10it/s]

MODEL_FP: af2305bdd0a7a91b
X_FP: bafdebb3d1c993b8
LOGITS_FP: 0be2031792c35dfc
PREDS_FP: 71b60f70dc4df314
model.training: False
param dtype: torch.float32
X dtype: torch.float32
device: mps:0


In [ ]:
meta = json.loads((SEQ_DIR / "meta.json").read_text())

t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
t_to_end_min_values = X_test[:, -1, t_to_end_min_idx].astype(int)

assert len(t_to_end_min_values) == len(all_labels) == len(all_probs)


In [31]:
import pandas as pd

df = pd.DataFrame({
    "t_to_end_min": t_to_end_min_values,
    "y_true": all_labels,
    "p_up": all_probs,
    "y_pred": all_preds,
})


In [32]:
import numpy as np
import json

df["tp"] = ((df.y_true==1) & (df.y_pred==1)).astype(int)
df["fp"] = ((df.y_true==0) & (df.y_pred==1)).astype(int)
df["tn"] = ((df.y_true==0) & (df.y_pred==0)).astype(int)
df["fn"] = ((df.y_true==1) & (df.y_pred==0)).astype(int)
df["correct"] = (df.y_true == df.y_pred).astype(int)

stats = df.groupby("t_to_end_min").agg(
    count=("correct","count"),
    tp=("tp","sum"),
    fp=("fp","sum"),
    tn=("tn","sum"),
    fn=("fn","sum"),
    accuracy=("correct","mean"),
    avg_p_up=("p_up","mean"),
    base_rate=("y_true","mean"),
).reset_index()

# -----------------------------
# Add all classification metrics
# -----------------------------
def safe_div(num, den):
    return np.where(den == 0, np.nan, num / den)

# Positive-class metrics (UP = 1)
stats["precision"] = safe_div(stats["tp"], stats["tp"] + stats["fp"])   # PPV
stats["recall"]    = safe_div(stats["tp"], stats["tp"] + stats["fn"])   # TPR / Sensitivity

stats["f1"] = safe_div(
    2 * stats["precision"] * stats["recall"],
    stats["precision"] + stats["recall"]
)

# Negative-class metrics (DOWN = 0)
stats["specificity"] = safe_div(stats["tn"], stats["tn"] + stats["fp"]) # TNR
stats["npv"]         = safe_div(stats["tn"], stats["tn"] + stats["fn"]) # Negative Predictive Value

# Optional but common:
stats["fpr"] = safe_div(stats["fp"], stats["fp"] + stats["tn"])         # False Positive Rate
stats["fnr"] = safe_div(stats["fn"], stats["fn"] + stats["tp"])         # False Negative Rate

# Percent formats (optional)
stats["accuracy_pct"] = stats["accuracy"] * 100
stats["precision_pct"] = stats["precision"] * 100
stats["recall_pct"] = stats["recall"] * 100
stats["f1_pct"] = stats["f1"] * 100
stats["specificity_pct"] = stats["specificity"] * 100
stats["npv_pct"] = stats["npv"] * 100

# -----------------------------
# Save to JSON
# -----------------------------
stats_json = stats.to_dict(orient="records")

OUT_DIR = RUN_DIR / "eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "per_t_to_end_min_stats.json"
out_path.write_text(json.dumps(stats_json, indent=2))

print(stats.to_string(index=False))
print(f"\nSaved per-minute stats to {out_path}")


 t_to_end_min  count    tp   fp    tn   fn  accuracy  avg_p_up  base_rate  precision   recall       f1  specificity      npv      fpr      fnr  accuracy_pct  precision_pct  recall_pct    f1_pct  specificity_pct   npv_pct
            1  31564 15433  248 15575  308  0.982385  0.511086   0.498701   0.984185 0.980433 0.982305     0.984327 0.980608 0.015673 0.019567     98.238500      98.418468   98.043326 98.230539        98.432661 98.060820
            2  31564 14730  958 14865 1011  0.937619  0.511765   0.498701   0.938934 0.935773 0.937351     0.939455 0.936319 0.060545 0.064227     93.761881      93.893422   93.577282 93.735085        93.945522 93.631897
            3  31564 14179 1511 14312 1562  0.902642  0.511828   0.498701   0.903697 0.900769 0.902230     0.904506 0.901600 0.095494 0.099231     90.264225      90.369662   90.076869 90.223028        90.450610 90.160010
            4  31564 13724 1921 13902 2017  0.875238  0.510374   0.498701   0.877213 0.871863 0.874530     0.878594 

## Calibration & Threshold

In [33]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calibration_curve(df_sub, bins=10, min_count=5):
    df_sub = df_sub.copy()
    df_sub["prob_bin"] = pd.cut(df_sub["p_up"], bins=bins)

    calib = df_sub.groupby("prob_bin").agg(
        avg_p=("p_up", "mean"),
        freq_up=("y_true", "mean"),
        count=("y_true", "count"),
    ).dropna()

    return calib[calib["count"] >= min_count]


# ----------------------------
# Calibration plots per t
# ----------------------------
OUT_DIR = RUN_DIR / "eval" / "calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

for t, g in df.groupby("t_to_end_min"):
    calib = calibration_curve(g, bins=10, min_count=5)
    if len(calib) < 2:
        continue

    total_n = len(g)

    plt.figure()

    # Model calibration curve
    plt.plot(
        calib["avg_p"],
        calib["freq_up"],
        marker="o",
        label="Model calibration"
    )

    # Perfect calibration reference
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="green",
        label="Perfect calibration"
    )

    # Annotate each point with absolute + percentage
    for _, row in calib.iterrows():
        pct = 100 * row["count"] / total_n
        plt.annotate(
            f"n={int(row['count'])}\n({pct:.1f}%)",
            (row["avg_p"], row["freq_up"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
        )

    # Axis ticks at 0.1
    ticks = np.linspace(0, 1, 11)
    plt.xticks(ticks)
    plt.yticks(ticks)
    plt.grid(True, which="both", linestyle="--", alpha=0.5)

    plt.xlabel("Average predicted probability")
    plt.ylabel("Empirical UP frequency")
    plt.title(f"Calibration curve (t_to_end_min={t}, n={total_n})")
    plt.legend(loc="lower right")

    plt.savefig(
        OUT_DIR / f"calibration_t{t}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.close()

# ------------------------------------
# 2) Threshold stats + plots per t
# ------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

THRESHOLDS = np.linspace(0.5, 0.9, 9)

BASE_DIR = RUN_DIR / "eval" / "threshold_analysis"
UP_DIR   = BASE_DIR / "up"
DN_DIR   = BASE_DIR / "down"
BASE_DIR.mkdir(parents=True, exist_ok=True)
UP_DIR.mkdir(parents=True, exist_ok=True)
DN_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------
# Build threshold tables
# --------------------------
rows_up = []
rows_dn = []

for t, g in df.groupby("t_to_end_min"):
    n_total = len(g)

    for tau in THRESHOLDS:
        # ---- UP: confident UP when p_up >= tau
        sel_up = g[g["p_up"] >= tau]
        if len(sel_up) > 0:
            rows_up.append({
                "t_to_end_min": int(t),
                "threshold": float(tau),
                "n_total": int(n_total),
                "n_samples": int(len(sel_up)),
                "coverage": float(len(sel_up) / n_total),
                "accuracy": float((sel_up["y_pred"] == sel_up["y_true"]).mean()),
                "base_rate_up": float(sel_up["y_true"].mean()),  # UP rate in selected set
            })

        # ---- DOWN: confident DOWN when p_up <= (1 - tau)
        sel_dn = g[g["p_up"] <= (1 - tau)]
        if len(sel_dn) > 0:
            # if we "act DOWN", predicted label is 0
            rows_dn.append({
                "t_to_end_min": int(t),
                "threshold": float(tau),
                "n_total": int(n_total),
                "n_samples": int(len(sel_dn)),
                "coverage": float(len(sel_dn) / n_total),
                "accuracy": float((sel_dn["y_true"] == 0).mean()),
                "base_rate_up": float(sel_dn["y_true"].mean()),  # should be low if DOWN is correct
            })

thr_up = pd.DataFrame(rows_up)
thr_dn = pd.DataFrame(rows_dn)

# --------------------------
# Plot helper (no extra funcs)
# --------------------------
for side, thr_df, out_dir in [
    ("UP", thr_up, UP_DIR),
    ("DOWN", thr_dn, DN_DIR),
]:
    if thr_df.empty:
        continue

    for t, g in thr_df.groupby("t_to_end_min"):
        g = g.sort_values("threshold")

        plt.figure()
        ax = plt.gca()

        # Accuracy (blue)
        line1, = ax.plot(
            g["threshold"],
            g["accuracy"],
            marker="o",
            color="blue",
            label="Accuracy given predicted probability ≥ τ"
        )

        # X axis label depends on side
        if side == "UP":
            ax.set_xlabel("Probability threshold (p_up ≥ τ)")
        else:
            ax.set_xlabel("Probability threshold (p_up ≤ 1 − τ)")

        ax.set_ylabel("Accuracy given predicted probability ≥ τ")

        # Coverage (green)
        ax2 = ax.twinx()
        line2, = ax2.plot(
            g["threshold"],
            g["coverage"] * 100,
            marker="s",
            linestyle="--",
            color="green",
            label="Coverage (%)"
        )
        ax2.set_ylabel("Coverage (%)")

        # Annotate absolute counts
        for _, row in g.iterrows():
            ax.annotate(
                f"{int(row['n_samples'])}",
                (row["threshold"], row["accuracy"]),
                textcoords="offset points",
                xytext=(0, 6),
                ha="center",
                fontsize=8,
            )

        ax.grid(True)
        plt.title(f"{side}: Accuracy & coverage vs threshold (t_to_end_min={t})")

        ax.legend(handles=[line1, line2], loc="lower right")

        plt.savefig(out_dir / f"accuracy_coverage_t{t}.png", dpi=150, bbox_inches="tight")
        plt.close()

# --------------------------
# Save summary.json (both)
# --------------------------
summary = {
    "thresholds": [float(x) for x in THRESHOLDS],
    "up": thr_up.to_dict(orient="records"),
    "down": thr_dn.to_dict(orient="records"),
}

(BASE_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print("Saved:", BASE_DIR / "summary.json")
print("UP plots  :", UP_DIR)
print("DOWN plots:", DN_DIR)


/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_34192/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = df_sub.groupby("prob_bin").agg(
/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_34192/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = df_sub.groupby("prob_bin").agg(
/var/folders/98/dyrftdkj6hl9_qxt7gl5_vjr0000gn/T/ipykernel_34192/1957432944.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the fut

Saved: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_legacy/eval/threshold_analysis/summary.json
UP plots  : /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_legacy/eval/threshold_analysis/up
DOWN plots: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/runs/LSTMClassifier_seq60_bs16_TrainShuffleTrue_legacy/eval/threshold_analysis/down
